# Matching team and member names

This notebook provides functions to match team names from the official team metadata (`team_meta_full.tsv`) with the corresponding team folders in `data/igem_team_websites/2022`. Each of these folders contains various `.txt` or `.html` files for a single team.

The main goal is to extract the raw attribution content (from either `attributions.txt` or `attributions.html`) for each team and compile it into a JSON file. The team names should be taken from `team_meta_full.tsv`. This JSON file will later be used to extract individual task attributions for each team member with LLMs. 

In the second part of this notebook, each team's roster is extracted from an API, and saved as a `.tsv` and JSON file. These names will later be given as part of the LLM prompts for extracting people and their tasks from raw attribution texts (see `9_llm_extract_names_and_task_descriptions.ipynb`).

Two resulting files from this notebook can be found on locations:
- `"../data/attributions/2022_attributions/raw_attributions_2022.json"`
- `"../data/attributions/2022_attributions/team_rosters_2022.json"`

In [1]:
import requests
import re
import difflib
import pandas as pd
from pathlib import Path
import json
from tqdm import tqdm
import subprocess

## 1. Matching Team Names

- matching webiste folder team names to team names in team meta
- extracting raw attributions for each team from their website folders

### 1.1 Save team names in lists 

In [2]:
# Save the 2022 team names from team_meta_full.tsv in a list
team_meta_df = pd.read_table(
    "../data/igem_scrapping_2025/team_meta_full.tsv",
    usecols=["Team", "Year", "TeamID", "Status"]
)

teams_meta_2022 = team_meta_df[team_meta_df["Year"] == 2022]["Team"].tolist()

In [3]:
teams_meta_2022

['AFCM-Egypt',
 'AHS_Peking',
 'ASIJ_Tokyo',
 'ASU',
 'Aachen',
 'Aalto-Helsinki',
 'Aboa',
 'Aix-Marseille',
 'Alma',
 'Anatolia_College_HS',
 'AshesiGhana',
 'Athens',
 'Austin_UTexas',
 'BFSU-ICUnited',
 'BGU_Israel',
 'BIT',
 'BIT-China',
 'BJEA_China',
 'BJWZ-China',
 'BNDS_China',
 'BNSC_China',
 'BNU-China',
 'BNUZH-China',
 'BOKU-Vienna',
 'BS_United_China',
 'BUCT',
 'BUCT-China',
 'Barcelona_UB',
 'Beijing_United',
 'Bilkent_UNAM',
 'Bio-Brussels',
 'Bioplus-China',
 'Bonn-Rheinbach',
 'BostonU_HW',
 'Bulgaria',
 'CAFA_China',
 'CAU_China',
 'CCA_San_Diego',
 'CCU_Taiwan',
 'CHINA-FAFU',
 'CPU_CHINA',
 'CPU_Nanjing',
 'CSMU_Taiwan',
 'CSU_CHINA',
 'CU-Boulder',
 'CUG-China',
 'CUHK-HongKong-SBS',
 'CUHKSZ',
 'CU_Egypt',
 'Calgary',
 'Cambridge',
 'Canton_HS',
 'Chalmers-Gothenburg',
 'CityU_HongKong',
 'City_of_London_UK',
 'Concordia-Montreal',
 'Cornell',
 'Costa_Rica',
 'Crete',
 'DKU',
 'DKU_China',
 'DNHS_SanDiego_CA',
 'DTU-Denmark',
 'DUT_China',
 'DeNovoCastrians',
 '

In [6]:
len(teams_meta_2022)

360

In [3]:
# function to save all folder names from data/igem_team_websites/2022 in a list (team_website_names_2022)
# for example, in the folder data/igem_team_websites/2022, there are folders like "aachen", "utokyo", etc., with each having attributions.txt files in them

def get_team_folder_names(base_path):
    path = Path(base_path)
    return [folder.name for folder in path.iterdir() if folder.is_dir()]

In [4]:
team_website_names_2022 = get_team_folder_names("../data/igem_team_websites/2022")

In [5]:
len(team_website_names_2022)

328

### 1.2 Match team meta and website folder names

In [7]:
# Function to set team names to lower case, remove dashes, remove all blank spaces
def clean_team_name(name):
        return name.lower().replace("-", " ").replace("_", " ").replace(" ", "").strip()

In [8]:
#Function to make a dictionary of name mappings between Team in team_meta_full.tsv and the team_website_names_2022 list 
def match_team_names(names_list_1, names_list_2):

    cleaned_2_map = {clean_team_name(name): name for name in names_list_2}

    name_map = {}
    for name1 in names_list_1:
        cleaned_name1 = clean_team_name(name1)
        matched_name = cleaned_2_map.get(cleaned_name1, "")
        name_map[name1] = matched_name

    return name_map

In [9]:
teams_names_map = match_team_names(teams_meta_2022, team_website_names_2022)

In [10]:
teams_names_map

{'AFCM-Egypt': 'afcm-egypt',
 'AHS_Peking': '',
 'ASIJ_Tokyo': 'asij-tokyo',
 'ASU': 'asu',
 'Aachen': 'aachen',
 'Aalto-Helsinki': 'aalto-helsinki',
 'Aboa': 'aboa',
 'Aix-Marseille': 'aix-marseille',
 'Alma': 'alma',
 'Anatolia_College_HS': 'anatolia-college-hs',
 'AshesiGhana': 'ashesighana',
 'Athens': 'athens',
 'Austin_UTexas': 'austin-utexas',
 'BFSU-ICUnited': 'bfsu-icunited',
 'BGU_Israel': 'bgu-israel',
 'BIT': 'bit',
 'BIT-China': 'bit-china',
 'BJEA_China': 'bjea-china',
 'BJWZ-China': 'bjwz-china',
 'BNDS_China': 'bnds-china',
 'BNSC_China': 'bnsc-china',
 'BNU-China': 'bnu-china',
 'BNUZH-China': 'bnuzh-china',
 'BOKU-Vienna': 'boku-vienna',
 'BS_United_China': 'bs-united-china',
 'BUCT': 'buct',
 'BUCT-China': 'buct-china',
 'Barcelona_UB': 'barcelona-ub',
 'Beijing_United': '',
 'Bilkent_UNAM': '',
 'Bio-Brussels': 'bio-brussels',
 'Bioplus-China': 'bioplus-china',
 'Bonn-Rheinbach': 'bonn-rheinbach',
 'BostonU_HW': 'bostonu-hw',
 'Bulgaria': 'bulgaria',
 'CAFA_China': 

In [ ]:
# Teams from the website folders that do not match any team from the team_meta_full.tsv
unmatched_names_from_websites = [k for k in teams_names_map if k == '']
len(unmatched_names_from_websites)

0

In [ ]:
# Teams from the team_meta_full.tsv that do not match any team from the website folders
unmatched_names_from_meta = [k for k, v in teams_names_map.items() if v == ""]
len(unmatched_names_from_meta)

32

In [22]:
# Dataframe of unmatched team names and their status from 
unmatched_df = team_meta_df[
    (team_meta_df["Year"] == 2022) &
    (team_meta_df["Team"].isin(unmatched_names_from_meta))
]

unmatched_df = unmatched_df[["Team", "Status"]]

unmatched_df

,Team,Status
20,AHS_Peking,withdrawn
398,Beijing_United,disqualified
434,Bilkent_UNAM,withdrawn
1016,East_China,disqualified
1362,Guangxi-U-China,disqualified
1383,HAGZGD-China,withdrawn
1516,HUS_United,disqualified
1802,ITESO_Guadalajara,accepted
2067,LZU-HS-China-A,disqualified
2068,LZU-HS-China-B,disqualified


In [23]:
unmatched_df["Status"].value_counts()

Status
disqualified    16
withdrawn       12
accepted         4
Name: count, dtype: int64


There are 32 more teams listed in `team_meta_full.tsv` than there are folders in `igem_team_websites/2022`. On the other hand, every team found in the `igem_team_websites/2022` folder also exists in `team_meta_full.tsv` (they have the same normalized names). Out of the 32 missing teams, 28 of them are either withdrawn or disqualified which explains the absence of their attributions folder. Four of them were accepted for the Jamboree, but they do not contain an attributions folder in `igem_team_websites/2022`. One of such teams is present in the 2022 survey, so its raw attributions will be extracted manually since it is more time-efficient for a single case.

### 1.3 Saving attributions data per each team

In [14]:
# Function to make a json file of all teams and their attributions txt files
no_attributions_teams = []

def collect_attributions_texts(base_path, output_json_path):
    base = Path(base_path)
    team_attributions = []

    for team_folder in sorted(base.iterdir(), key=lambda x: x.name.lower()):
        if not team_folder.is_dir():
            continue

        attributions_file = team_folder / "attributions.txt"
        fallback_file = team_folder / "attributions.html.txt"
        # some teams have attributions.html.txt files instead of attributions.txt

        if attributions_file.exists():
            file_to_read = attributions_file
        elif fallback_file.exists():
            file_to_read = fallback_file
        else:
            print(f"No attributions file found for team: {team_folder.name}")
            no_attributions_teams.append(team_folder.name)
            continue

        try:
            content = file_to_read.read_text(encoding="utf-8").strip()
            team_attributions.append({
                "teamName": team_folder.name,
                "attributionsText": content
            })
        except Exception as e:
            print(f"Error reading {file_to_read.name} for team {team_folder.name}: {e}")

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(team_attributions, f, ensure_ascii=False, indent=2)

    print(f"Raw team attributions website texts saved to {output_json_path}")


In [15]:
collect_attributions_texts(
    base_path="../data/igem_team_websites/2022",
    output_json_path="../data/attributions/2022_attributions/raw_attributions_2022.json"
)

No attributions file found for team: bgu-israel
No attributions file found for team: bnds-china
No attributions file found for team: city-of-london-uk
No attributions file found for team: fudan
No attributions file found for team: gdsyzx
No attributions file found for team: hku-hongkong
No attributions file found for team: iit-roorkee
No attributions file found for team: nju-china
No attributions file found for team: patras-medicine
No attributions file found for team: peking
No attributions file found for team: queens-canada
No attributions file found for team: stanford
No attributions file found for team: sysu-china
No attributions file found for team: thailand-ris
No attributions file found for team: tongji-software
No attributions file found for team: vilnius-lithuania
No attributions file found for team: yale
Raw team attributions website texts saved to ../data/attributions/2022_attributions/raw_attributions_2022.json


In [16]:
no_attributions_teams

['bgu-israel',
 'bnds-china',
 'city-of-london-uk',
 'fudan',
 'gdsyzx',
 'hku-hongkong',
 'iit-roorkee',
 'nju-china',
 'patras-medicine',
 'peking',
 'queens-canada',
 'stanford',
 'sysu-china',
 'thailand-ris',
 'tongji-software',
 'vilnius-lithuania',
 'yale']

In [ ]:
# Teams with no attribution files

team_meta_2022 = team_meta_df[team_meta_df['Year'] == 2022].copy() #so the names are the same as they were in 2022

team_meta_2022['Team_normalized'] = team_meta_2022['Team'].apply(clean_team_name)

normalized_no_attr = [clean_team_name(name) for name in no_attributions_teams]

df_no_attr = team_meta_2022[
    team_meta_2022['Team_normalized'].isin(normalized_no_attr)
].drop_duplicates(subset=['Team_normalized'])

df_no_attr_status = df_no_attr[['Team', 'Status']]

df_no_attr_status


,Team,Status
225,BGU_Israel,accepted
296,BNDS_China,accepted
781,City_of_London_UK,accepted
1189,Fudan,accepted
1213,GDSYZX,accepted
1448,HKU_HongKong,accepted
1772,IIT_Roorkee,accepted
2550,NJU-China,accepted
2933,Patras_Medicine,accepted
2946,Peking,accepted


There are 17 teams that do not have an `attributions.txt` or `attributions.html` file, but all of them were officially accepted. After checking their GitLab repositories, it was found that all of them do contain attributions information, although most of it is written in JavaScript. This explains why Rathin’s previous text extraction process skipped these teams. Since only 5 of such teams also appear in the survey, their raw attribution text, as well as each member’s task descriptions, will be extracted manually.


In [19]:
# Replace team names in the json file with the correct ones from the name map (gathered from team meta)

with open("../data/attributions/2022_attributions/raw_attributions_2022.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Take the keys from teams_names_map as correct name formats
reversed_map = {v: k for k, v in teams_names_map.items() if v}

for entry in data:
    original_name = entry["teamName"]
    corrected_name = reversed_map.get(original_name, original_name)
    entry["teamName"] = corrected_name

# Overwrite the original json file
with open("../data/attributions/2022_attributions/raw_attributions_2022.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [20]:
# Read raw attributions json file and create a DataFrame
with open("../data/attributions/2022_attributions/raw_attributions_2022.json", "r", encoding="utf-8") as f:
    attributions_data = json.load(f)
raw_attributions_df = pd.DataFrame(attributions_data)

raw_attributions_df

,teamName,attributionsText
0,Aachen,Project\nDescription Contribution Engineering ...
1,Aalto-Helsinki,Home\nProject\nProject\nDescription\nImplement...
2,Aboa,Home\nAbout us\nTeam members\nAttributions\nCo...
3,AFCM-Egypt,Home\nTeam\nContribution Attributions Team mem...
4,Aix-Marseille,Aix-Marseille\nHome\nTeam\nTeam\nAttributions\...
...,...,...
306,YkPaO,ProTeen-fessionals\nHome\nTeam\nTeam\nAttribut...
307,Zhejiang_United,MaiaPotion\nAwards\nEducation\nEntrepreneurshi...
308,ZJU-China,TEAM\nMember\nAttributions\nPartnership\nColla...
309,ZJUintl-China,Home\nTeam\nTeam\nAttributions\nProject\nDescr...


## 2. Extracting Member Names and Tasks

- extracting member names from team roster API

In [ ]:
# Save the 2022 team names and their IDs from team_meta_full.tsv in a list of team dictionaries (to be used in roster API calls)
# Each dictionary will have the keys "teamName" and "teamID" 

teams_2022_df = team_meta_df[team_meta_df["Year"] == 2022]

teams_names_ids_2022 = teams_2022_df.rename(columns={"Team": "teamName", "TeamID": "teamID", "Year": "year"})[
    ["teamName", "teamID", "year"]
].to_dict(orient="records")

teams_names_ids_2022

[{'teamName': 'AFCM-Egypt', 'teamID': 4140, 'year': 2022},
 {'teamName': 'AHS_Peking', 'teamID': 4396, 'year': 2022},
 {'teamName': 'ASIJ_Tokyo', 'teamID': 4334, 'year': 2022},
 {'teamName': 'ASU', 'teamID': 4455, 'year': 2022},
 {'teamName': 'Aachen', 'teamID': 4138, 'year': 2022},
 {'teamName': 'Aalto-Helsinki', 'teamID': 4159, 'year': 2022},
 {'teamName': 'Aboa', 'teamID': 4207, 'year': 2022},
 {'teamName': 'Aix-Marseille', 'teamID': 4189, 'year': 2022},
 {'teamName': 'Alma', 'teamID': 4166, 'year': 2022},
 {'teamName': 'Anatolia_College_HS', 'teamID': 4503, 'year': 2022},
 {'teamName': 'AshesiGhana', 'teamID': 4186, 'year': 2022},
 {'teamName': 'Athens', 'teamID': 4294, 'year': 2022},
 {'teamName': 'Austin_UTexas', 'teamID': 4342, 'year': 2022},
 {'teamName': 'BFSU-ICUnited', 'teamID': 4400, 'year': 2022},
 {'teamName': 'BGU_Israel', 'teamID': 4120, 'year': 2022},
 {'teamName': 'BIT', 'teamID': 4128, 'year': 2022},
 {'teamName': 'BIT-China', 'teamID': 4232, 'year': 2022},
 {'teamNa

In [ ]:
# Save the 2022 roster data to a tsv file
# API call to get the team roster for 2022 (because the team roster tsv file that we have now is anonymized)
# Reusing some code from get_team_rosters-checkpoint.ipynb 

skipped_entries = []

def get_roster(team):
    year = team.get("year")
    team_id = team.get("teamID")
    team_name = team.get("teamName")
    url = f"https://api.igem.org/v1/teams/{team_id}/roster"

    try:
        r = requests.get(url)
        r.raise_for_status()
        data = r.json()
        records = []

        role_map = {
            "student": "Student Member",
            "student-leader": "Student Leader",
            "instructor": "Instructor",
            "primary-pi": "Primary PI",
            "secondary-pi": "Secondary PI"
        }

        for entry in data:
            roster_uuid = entry.get("uuid", "")
            role_key = entry.get("role", "").lower()
            role = role_map.get(role_key, role_key.replace("-", " ").title())

            member = entry.get("member") or {}
            user = member.get("user") or {}

            records.append({
                "FullName": user.get("publicName", ""),
                "TeamID": team_id,
                "Team": team_name,
                "Year": year,
                "Role": role,
                "Username": member.get("username", ""),
                "RosterUUID": roster_uuid,
                "UserUUID": member.get("uuid", ""),
                "UserID": user.get("id", ""),    
            })

        return records

    except Exception as e:
        print(f"Error for team {team_name} ({team_id}): {e}")
        skipped_entries.append({
            "Year": year,
            "TeamID": team_id,
            "Team": team_name,
            "Error": str(e),
            "RawEntry": None
        })
        return []


# Execution
all_2022_records = []
for team in tqdm(teams_names_ids_2022, desc="Fetching rosters"):
    all_2022_records.extend(get_roster(team))

df_roster = pd.DataFrame(all_2022_records)
df_roster.sort_values(["Team", "Role"], inplace=True)
df_roster.to_csv("../data/attributions/2022_attributions/team_rosters_2022.tsv", sep="\t", index=False)
print(f"Saved team_rosters_2022.tsv with {len(df_roster)} rows")

Fetching rosters: 100%|██████████| 360/360 [00:23<00:00, 15.36it/s]

Saved team_rosters_2022.tsv with 7762 rows


In [26]:
df_roster = pd.read_table("../data/attributions/2022_attributions/team_rosters_2022.tsv")

In [ ]:
# Save Teams and FullNames in a dictionary (later used for LLM prompts)
roster_2022_dict = (
    df_roster
    .groupby("Team")["FullName"]
    .apply(list)
    .to_dict()
)

roster_2022_dict

In [ ]:
# Save the dictionary as a JSON file

roster_output_path = "../data/attributions/2022_attributions/team_rosters_2022.json"

with open(roster_output_path, "w", encoding="utf-8") as f:
    json.dump(roster_2022_dict, f, ensure_ascii=False, indent=2)